# 34 — Error analysis y diagnósticos por segmento

Analiza el modelo ya seleccionado exclusivamente sobre la mitad de VALIDATION reservada para selección. No cambia el modelo y no consume TEST.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (MODEL_DIR / "selected_model.json").exists(), "Ejecuta 33 primero."
from argos.models.error_analysis import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


In [ ]:
from argos.models.error_analysis import build_error_analysis

result = build_error_analysis(
    root=paths.root,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.segments.head(50)


In [ ]:
result.score_deciles

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("34 ✅ PASS" if result.ready_for_next_notebook else "34 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
assert result.ready_for_next_notebook
